# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器

本筆記本專為 **Google Colab VS Code Extension** 設計：
- 在本地 VS Code 編輯代碼
- 運算核心切換為 **Google Colab 遠端運算**（本機 CPU 使用率 0%）
- 調參目標：**LMR 第一階段核心六軸（BASE_OFFSET + HIST_SCALE + CUTNODE + TTCAP + MOVECOUNT + TABLE_SCALE）深度調參（80 輪 x 150 局 = 12,000 局）**

### 💡 在 VS Code 中連接 Colab 核心步驟：
1. 點擊本編輯器右上角的 **「Select Kernel（選取核心）」**
2. 選擇 **「Colab」**
3. 登入 Google 帳號授權，選取一個運行階段（Runtime）
4. 依序執行下方 Cell 即可開始雲端自動化調參！

In [18]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python --version

CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.15


In [19]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout {branch}
!git pull origin {branch}
!pip install -q numba chess numpy

/content/chess_numba
M	tune_search/current_params.txt
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.15 KiB | 1.15 MiB/s, done.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
   68b5953..bff6b7b  feat-initial-bitboard-setup -> origin/feat-initial-bitboard-setup
Updating 68b5953..bff6b7b
Fast-forward
 .gitignore                           |  2 ++
 tune_colab_runner.ipynb              | 42 +++++++++++++++++++++++++++---------
 tune_search/search_param_registry.py |  2 +-
 3 files changed, 35 insertions(+), 11 deletions(-)


In [ ]:
# 3. 測試環境與常數載入
import sys
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

target_params = [
    "LMR_BASE_OFFSET",
    "LMR_HISTORY_SCALE",
    "LMR_CUTNODE_BONUS",
    "LMR_TTCAPTURE_BONUS",
    "LMR_MOVECOUNT_FACTOR",
    "LMR_TABLE_SCALE_PERCENT"
]
print("✅ 成功載入！LMR 第一階段核心六軸基準預設值：")
print(get_default_params(names=target_params))


In [ ]:
# 4. 啟動 L1 SPSA LMR 第一階段核心六軸雲端對抗調參（80 輪 x 150 局 = 12,000 局）
import os
cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

!python -m tune_search.search_tuner \
  --params LMR_BASE_OFFSET,LMR_HISTORY_SCALE,LMR_CUTNODE_BONUS,LMR_TTCAPTURE_BONUS,LMR_MOVECOUNT_FACTOR,LMR_TABLE_SCALE_PERCENT \
  --backend inprocess \
  --scale-mode integer \
  --iter 80 \
  --games 150 \
  --nodes 50000 \
  --concurrency {concurrency} \
  --adjudicate \
  --log tune_search/logs/lmr_p1_deep_l1.jsonl \
  --state tune_search/states/lmr_p1_deep_l1.json


In [ ]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後半段平均產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/lmr_p1_deep_l1.jsonl --tail 0.5 --out tune_search/candidate_lmr_p1_deep.json
print("\n=== 候選參數 JSON 內容 ===")
!cat tune_search/candidate_lmr_p1_deep.json


In [ ]:
# 6. 直接在雲端進行 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局）
!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p1_deep.json \
  --nodes 300000 \
  --games 600 \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_lmr_p1_deep.json
